# Foreign Whispers — TTS Dubbing (Colab Direct)

Runs TTS synthesis **directly on Colab GPU** using the local Chatterbox server at `localhost:8020`.
Outputs a ready-to-stitch `.wav` per video.

**Before running:**
1. Ensure the Chatterbox model is loaded (run setup notebook cells 1–3)
2. Upload the translation JSON files when prompted in Cell 2

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────
!pip install pydub tqdm requests scipy -q
print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Upload translation JSON files ─────────────────────────────
# Upload: pipeline_data/api/translations/argos/Rob Reiner: The 60 Minutes Interview.json
#         pipeline_data/api/translations/argos/Military Drones: 60 Minutes Full Episodes.json
from google.colab import files as colab_files
print('Upload the translation JSON files now...')
uploaded = colab_files.upload()
print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
# ── Cell 3: Core TTS engine ────────────────────────────────────────────
import json, io, time, os, pathlib, tempfile, logging
import requests
from pydub import AudioSegment
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('fw_tts')

CHATTERBOX_URL = 'http://localhost:8020'  # local Chatterbox on Colab
MAX_WORKERS = 5       # keep below GPU queue limit
TIMEOUT_S   = 120     # per-segment timeout
MAX_RETRIES = 2       # retries per segment on transient failure


def check_chatterbox():
    """Verify Chatterbox is up before starting."""
    try:
        r = requests.post(
            f'{CHATTERBOX_URL}/v1/audio/speech',
            json={'input': 'prueba', 'response_format': 'wav'},
            timeout=30,
        )
        if r.status_code == 200:
            print(f'✓ Chatterbox alive at {CHATTERBOX_URL} ({len(r.content)} bytes warmup)')
            return True
        print(f'✗ Chatterbox returned {r.status_code}')
    except Exception as e:
        print(f'✗ Chatterbox unreachable: {e}')
    return False


def synthesize_one(text: str) -> bytes | None:
    """Synthesize one text chunk, with retries. Returns WAV bytes or None."""
    if not text or not text.strip():
        return None
    for attempt in range(MAX_RETRIES + 1):
        try:
            r = requests.post(
                f'{CHATTERBOX_URL}/v1/audio/speech',
                json={'input': text.strip(), 'response_format': 'wav'},
                timeout=TIMEOUT_S,
            )
            if r.status_code == 200:
                return r.content
            log.warning('Segment HTTP %d (attempt %d/%d)', r.status_code, attempt+1, MAX_RETRIES+1)
        except requests.Timeout:
            log.warning('Timeout on attempt %d/%d', attempt+1, MAX_RETRIES+1)
        except Exception as e:
            log.warning('Error on attempt %d/%d: %s', attempt+1, MAX_RETRIES+1, e)
        if attempt < MAX_RETRIES:
            time.sleep(1)
    return None


def run_tts_for_video(json_path: str, output_wav: str, workers: int = MAX_WORKERS):
    """Full TTS pipeline for one video. Saves assembled WAV to output_wav."""
    print(f'\n{'='*60}')
    print(f'Processing: {pathlib.Path(json_path).stem}')
    print(f'Output:     {output_wav}')
    print(f'{'='*60}')

    with open(json_path) as f:
        data = json.load(f)
    segments = data.get('segments', [])
    if not segments:
        print('No segments found in JSON')
        return

    print(f'Total segments : {len(segments)}')
    print(f'Video duration : {segments[-1]["end"]:.0f}s ({segments[-1]["end"]/60:.1f} min)')
    print(f'Workers        : {workers}')

    # ── Phase 1: GPU synthesis ─────────────────────────────────────────
    raw_wav_map: dict[int, bytes | None] = {}
    success_count = 0
    fail_count    = 0

    print(f'\nPhase 1: Synthesizing {len(segments)} segments...')
    phase1_bar = tqdm(total=len(segments), unit='seg', ncols=80,
                      bar_format='{l_bar}{bar}| {n}/{total} [{elapsed}<{remaining}, {rate_fmt}]')

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {
            pool.submit(synthesize_one, seg['text']): idx
            for idx, seg in enumerate(segments)
        }
        for fut in as_completed(futures):
            idx = futures[fut]
            try:
                wav_bytes = fut.result()
            except Exception as e:
                log.error('Segment %d raised: %s', idx, e)
                wav_bytes = None
            raw_wav_map[idx] = wav_bytes
            if wav_bytes:
                success_count += 1
            else:
                fail_count += 1
            phase1_bar.update(1)
            phase1_bar.set_postfix(ok=success_count, fail=fail_count)

    phase1_bar.close()
    pct = success_count / len(segments) * 100
    status = '✓' if pct >= 95 else ('⚠' if pct >= 70 else '✗')
    print(f'{status} Synthesis: {success_count}/{len(segments)} ({pct:.1f}%) succeeded, {fail_count} failed')

    if success_count == 0:
        print('ERROR: 0 segments synthesized — check Chatterbox is running (cell 3 healthcheck)')
        return

    # ── Phase 2: Assemble with timing ─────────────────────────────────
    print(f'\nPhase 2: Assembling timed audio...')
    total_ms  = max(int(s['end'] * 1000) for s in segments)
    combined  = AudioSegment.empty()
    cursor_ms = 0
    clipped   = 0
    padded    = 0

    for idx, seg in enumerate(tqdm(segments, unit='seg', ncols=80)):
        start_ms = max(0, int(seg['start'] * 1000))

        if idx + 1 < len(segments):
            next_start_ms = max(start_ms, int(segments[idx+1]['start'] * 1000))
            slot_ms = next_start_ms - start_ms
        else:
            slot_ms = max(0, total_ms - start_ms)

        if slot_ms <= 0:
            continue

        wav_bytes = raw_wav_map.get(idx)
        if wav_bytes is None:
            seg_audio = AudioSegment.silent(duration=slot_ms)
            padded += 1
        else:
            seg_audio = AudioSegment.from_wav(io.BytesIO(wav_bytes))
            if len(seg_audio) > slot_ms:
                seg_audio = seg_audio[:slot_ms]
                clipped += 1

        if start_ms > cursor_ms:
            combined += AudioSegment.silent(duration=start_ms - cursor_ms)
            cursor_ms = start_ms
        combined += seg_audio
        cursor_ms += len(seg_audio)

    if cursor_ms < total_ms:
        combined += AudioSegment.silent(duration=total_ms - cursor_ms)

    print(f'  Duration: {len(combined)/1000:.1f}s | Clipped: {clipped} segs | Silence-padded: {padded} segs')

    # ── Phase 3: Export ────────────────────────────────────────────────
    print(f'\nPhase 3: Exporting WAV...')
    combined.export(output_wav, format='wav')
    size_mb = pathlib.Path(output_wav).stat().st_size / 1e6
    print(f'✓ Saved {output_wav} ({size_mb:.1f} MB)')
    return output_wav


# Healthcheck
check_chatterbox()

In [ ]:
# ── Cell 4: Rob Reiner TTS (~8 min) ───────────────────────────────────
import os
rob_title = 'Rob Reiner: The 60 Minutes Interview'
rob_json  = f'/content/{rob_title}.json'
rob_wav   = f'/content/{rob_title}.wav'

if not os.path.exists(rob_json):
    print(f'ERROR: {rob_json} not found — upload it in Cell 2')
else:
    run_tts_for_video(rob_json, rob_wav, workers=MAX_WORKERS)

In [ ]:
# ── Cell 5: Military Drones TTS (~30 min) ─────────────────────────────
mil_title = 'Military Drones: 60 Minutes Full Episodes'
mil_json  = f'/content/{mil_title}.json'
mil_wav   = f'/content/{mil_title}.wav'

if not os.path.exists(mil_json):
    print(f'ERROR: {mil_json} not found — upload it in Cell 2')
else:
    run_tts_for_video(mil_json, mil_wav, workers=MAX_WORKERS)

In [ ]:
# ── Cell 6: Download WAVs ─────────────────────────────────────────────
from google.colab import files as colab_files
import os

for wav_path in [rob_wav, mil_wav]:
    if os.path.exists(wav_path):
        size_mb = os.path.getsize(wav_path) / 1e6
        print(f'Downloading {os.path.basename(wav_path)} ({size_mb:.1f} MB)...')
        colab_files.download(wav_path)
    else:
        print(f'MISSING: {wav_path} — did the TTS cell complete?')

## After Download

Place the downloaded WAVs at:
```
pipeline_data/api/tts_audio/chatterbox/c-e5604bc/Rob Reiner: The 60 Minutes Interview.wav
pipeline_data/api/tts_audio/chatterbox/c-e5604bc/Military Drones: 60 Minutes Full Episodes.wav
```
Then trigger stitch:
```bash
curl -X POST 'http://localhost:8080/api/stitch/DLeBquj8LKI?config=c-e5604bc'
curl -X POST 'http://localhost:8080/api/stitch/IiBKsv-D64M?config=c-e5604bc'
```